In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
# Data and model paths
ROOT = Path.cwd().parent
DATA_DIR = ROOT / 'ml' / 'data'
MODELS_DIR = ROOT / 'include' / 'models'

In [3]:
# Model parameters
EPOCHS = 100
LEARNING_RATE = 0.05
BATCH_SIZE = 1024

In [4]:
# Load the dataset
train_data = pd.read_csv(DATA_DIR / 'MNIST' / 'pre_train.csv')
test_data = pd.read_csv(DATA_DIR / 'MNIST' / 'test.csv')

print(f'Train data shape: {train_data.shape}')
print(f'Test data shape: {test_data.shape}')

Train data shape: (20000, 197)
Test data shape: (10000, 197)


In [5]:
# Extract features and labels
X_train = train_data.drop('label', axis=1).to_numpy()
X_test = test_data.drop('label', axis=1).to_numpy()

y_train = train_data['label'].to_numpy()
y_test = test_data['label'].to_numpy()

print(f'Train Class distribution:\n{pd.Series(y_train).value_counts().sort_index()}'); print()
print(f'Test Class distribution:\n{pd.Series(y_test).value_counts().sort_index()}')

Train Class distribution:
0    1972
1    2250
2    1997
3    2040
4    1950
5    1804
6    1965
7    2084
8    1950
9    1988
Name: count, dtype: int64

Test Class distribution:
0     986
1    1125
2     999
3    1020
4     975
5     902
6     982
7    1042
8     975
9     994
Name: count, dtype: int64


In [6]:
# Convert data to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [7]:
# Define the NN model
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(196, 32), nn.ReLU(),   # Input layer (14 x 14 = 196 pixels) to hidden layer 1 (32 neurons)
            nn.Linear(32, 16), nn.ReLU(),    # Hidden layer 1 (32 neurons) to hidden layer 2 (16 neurons)
            nn.Linear(16, 10)                # Hidden layer 2 (16 neurons) to output layer (10 classes)
        )

    def forward(self, x):
        return self.network(x)


model = MNISTNet()

In [8]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [9]:
# Training loop
print("Training...")

dataset = torch.utils.data.TensorDataset(X_train, y_train)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    epoch_loss = total_loss / len(dataset)

    print(f"Epoch [{epoch+1:2d}/{EPOCHS}] Loss: {epoch_loss:.4f}")

Training...
Epoch [ 1/100] Loss: 2.3079
Epoch [ 2/100] Loss: 2.2986
Epoch [ 3/100] Loss: 2.2880
Epoch [ 4/100] Loss: 2.2750
Epoch [ 5/100] Loss: 2.2588
Epoch [ 6/100] Loss: 2.2365
Epoch [ 7/100] Loss: 2.2043
Epoch [ 8/100] Loss: 2.1592
Epoch [ 9/100] Loss: 2.1009
Epoch [10/100] Loss: 2.0309
Epoch [11/100] Loss: 1.9509
Epoch [12/100] Loss: 1.8598
Epoch [13/100] Loss: 1.7539
Epoch [14/100] Loss: 1.6326
Epoch [15/100] Loss: 1.4997
Epoch [16/100] Loss: 1.3651
Epoch [17/100] Loss: 1.2396
Epoch [18/100] Loss: 1.1305
Epoch [19/100] Loss: 1.0388
Epoch [20/100] Loss: 0.9634
Epoch [21/100] Loss: 0.9011
Epoch [22/100] Loss: 0.8492
Epoch [23/100] Loss: 0.8059
Epoch [24/100] Loss: 0.7680
Epoch [25/100] Loss: 0.7354
Epoch [26/100] Loss: 0.7064
Epoch [27/100] Loss: 0.6808
Epoch [28/100] Loss: 0.6575
Epoch [29/100] Loss: 0.6366
Epoch [30/100] Loss: 0.6169
Epoch [31/100] Loss: 0.5992
Epoch [32/100] Loss: 0.5828
Epoch [33/100] Loss: 0.5679
Epoch [34/100] Loss: 0.5535
Epoch [35/100] Loss: 0.5407
Epoch [3

In [10]:
# Evaluate the model
model.eval()

with torch.no_grad():
    y_pred = model(X_test)
    y_pred_classes = torch.argmax(y_pred, dim=1)

    accuracy = accuracy_score(y_test, y_pred_classes)
    print(f"\nTest Accuracy: {accuracy:.4f}")


Test Accuracy: 0.9026


In [11]:
class MNISTNetExport(nn.Module):
    def __init__(self):
        super(MNISTNetExport, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(196, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 10), nn.Softmax(dim=1)  # add Softmax layer to get output probabilities
        )

    def forward(self, x):
        return self.network(x)

export_model = MNISTNetExport()
export_model.load_state_dict(model.state_dict())

export_model.eval()
with torch.no_grad():
    y_pred_export = export_model(X_test)
    y_pred_classes_export = torch.argmax(y_pred_export, dim=1)

    accuracy_export = accuracy_score(y_test, y_pred_classes_export)
    print(f"\nTest Accuracy (Export Model): {accuracy_export:.4f}")


Test Accuracy (Export Model): 0.9026


In [12]:
from aifes import pytorch2aifes

pytorch2aifes.convert_to_fnn_f32_express(export_model, str(MODELS_DIR / 'MNISTNet'))